# Physics-Informed Neural Networks for High-Temperature Turbomachinery Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedsoczfod-hue/pinn-thermo-dynamics/blob/main/notebooks/demo.ipynb)
![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue.svg)
![PyTorch 2.x](https://img.shields.io/badge/PyTorch-2.x-ee4c2c.svg)
![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)

This interactive tutorial demonstrates how to solve and calibrate **Physics-Informed Neural Networks (PINNs)** for high-temperature aerothermal blade dynamics, combining second-order Autograd, **Modified MLP with Fourier Feature Embeddings**, and **Adaptive Loss Weighting (Wang et al., 2021)**.

## 1. Mathematical Formulation

The transient aerothermal heat conduction across a cooled gas turbine blade wall is modeled by:

$$\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2} + Q(x, t), \quad x \in [0, 1], \; t \in [0, 1]$$

where:
- $T(x, t)$ is the non-dimensional temperature field.
- $\alpha = \frac{k}{\rho c_p}$ is the thermal diffusivity.
- $Q(x, t)$ is the aerothermal heat source / film-cooling thermal sink.
- Boundary conditions: Hot gas surface ($x=0$) and internal coolant channel ($x=1$).

In [ ]:
# Google Colab Setup (Automated environment bootstrap)
import os, sys
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/mohammedsoczfod-hue/pinn-thermo-dynamics.git
    %cd pinn-thermo-dynamics
    !pip install -r requirements.txt
    sys.path.append(os.getcwd())
else:
    # Local environment
    sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt
from physics.pde_residuals import ManufacturedHeatConduction, HeatConductionPDE, compute_pde_residuals
from physics.boundary_conds import BoundaryConditionManager, CollocationSampler
from models.networks import build_network
from models.pinn_solver import PINNSolver
from utils.metrics import compute_metrics
from utils.plotting import plot_solution_tripanel, plot_loss_convergence, set_academic_style

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Initialize Analytical Manufactured Benchmark

We instantiate an exact analytical solution $T^*(x, t)$ with true thermal diffusivity $\alpha = 0.05$ and thermal decay $\lambda = 0.5$.

In [ ]:
alpha_true = 0.05
decay_lambda = 0.5
benchmark = ManufacturedHeatConduction(alpha_true=alpha_true, decay_lambda=decay_lambda)
bc_manager = BoundaryConditionManager(benchmark=benchmark)
sampler = CollocationSampler(bc_manager=bc_manager, device=device, seed=42)

# Create visual evaluation grid
x_eval = np.linspace(0.0, 1.0, 100)
t_eval = np.linspace(0.0, 1.0, 100)
X_mesh, T_mesh = np.meshgrid(x_eval, t_eval)
T_exact = benchmark.exact_temperature(X_mesh, T_mesh)
print(f"Analytical temperature range: [{T_exact.min():.3f}, {T_exact.max():.3f}]")

## 3. Forward Simulation: Training the Modified MLP PINN

We train a **Modified MLP** with transformer-inspired residual gating to solve the forward problem.

In [ ]:
# Build Forward PDE and Network
pde_forward = HeatConductionPDE(alpha=alpha_true, is_inverse=False, decay_lambda=decay_lambda).to(device)
model_forward = build_network(
    arch_type="modified_mlp",
    in_dim=2,
    out_dim=1,
    hidden_dim=64,
    num_layers=4,
    activation="tanh",
).to(device)

solver_forward = PINNSolver(
    model=model_forward,
    pde=pde_forward,
    sampler=sampler,
    learning_rate=1e-3,
    use_adaptive_weights=True,
    device=device,
)

print("Training Forward PINN for 1000 epochs...")
for epoch in range(1001):
    metrics = solver_forward.train_step(epoch=epoch, n_pde=2000, n_ic=100, n_bc=100)
    solver_forward.history["loss_total"].append(metrics["loss_total"])
    solver_forward.history["loss_pde"].append(metrics["loss_pde"])
    solver_forward.history["loss_bc"].append(metrics["loss_bc"])
    solver_forward.history["loss_ic"].append(metrics["loss_ic"])
    solver_forward.history["loss_data"].append(metrics["loss_data"])
    if epoch % 200 == 0:
        l2_err = solver_forward.evaluate_relative_l2(100, 100)
        print(f"Epoch {epoch:4d} | Loss: {metrics['loss_total']:.2e} | Relative L2 Error: {l2_err:.3e}")

### Forward Verification: Tri-Panel Comparison

In [ ]:
model_forward.eval()
with torch.no_grad():
    coords = torch.tensor(np.column_stack([X_mesh.ravel(), T_mesh.ravel()]), dtype=torch.float32, device=device)
    T_pred = model_forward(coords).cpu().numpy().reshape(X_mesh.shape)

fig = plot_solution_tripanel(X_mesh, T_mesh, T_exact, T_pred, title_suffix="(Modified MLP)")
plt.show()

metrics_eval = compute_metrics(T_pred, T_exact)
print(f"Final Relative L2 Error: {metrics_eval['relative_l2_error']:.3e}")
print(f"Root Mean Square Error : {metrics_eval['rmse']:.3e}")

## 4. Inverse Problem: System Identification & Diffusivity Estimation

In this experiment, the thermal diffusivity $\alpha$ is **unknown**. We supply the network with sparse, noisy thermocouple sensor observations, starting from an erroneous initial guess $\alpha_0 = 0.02$ (true is $0.05$).

In [ ]:
# Inverse PDE and Network
pde_inverse = HeatConductionPDE(
    alpha=alpha_true,
    is_inverse=True,
    decay_lambda=decay_lambda,
    initial_guess_alpha=0.02,
).to(device)

model_inverse = build_network(
    arch_type="modified_mlp",
    in_dim=2,
    out_dim=1,
    hidden_dim=64,
    num_layers=4,
    activation="tanh",
).to(device)

solver_inverse = PINNSolver(
    model=model_inverse,
    pde=pde_inverse,
    sampler=sampler,
    learning_rate=1e-3,
    use_adaptive_weights=True,
    warmup_epochs=250,
    device=device,
)

print("Training Inverse PINN (Estimating unknown thermal diffusivity alpha)...")
alpha_history = []
for epoch in range(1001):
    metrics = solver_inverse.train_step(epoch=epoch, n_pde=2000, n_ic=100, n_bc=100, n_data=150, data_noise_std=0.005)
    alpha_history.append(metrics["alpha_learned"])
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Alpha Learned: {metrics['alpha_learned']:.5f} (True: {alpha_true}) | Error: {metrics['alpha_error_pct']:.2f}%")

### Parameter Convergence Plot

In [ ]:
set_academic_style()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alpha_history, label=r"Discovered Diffusivity $\hat{\alpha}$", color="#1f77b4", linewidth=2.0)
ax.axhline(alpha_true, label=rf"True Diffusivity $\alpha={alpha_true}$", color="#d62728", linestyle="--", linewidth=1.8)
ax.set_xlabel("Optimization Epoch")
ax.set_ylabel(r"Diffusivity $\alpha$ [$m^2/s$]")
ax.set_title("Inverse System Identification: Parameter Convergence")
ax.legend(frameon=True)
plt.show()

learned_final = pde_inverse.alpha.item()
pct_error = abs(learned_final - alpha_true) / alpha_true * 100.0
print(f"True Alpha: {alpha_true:.5f}")
print(f"Final Discovered Alpha: {learned_final:.5f}")
print(f"Final Parameter Error: {pct_error:.3f}%")